# Numerical Modeling and Optimization of the Photoresponse of a Monolayer MoS2 Photodetector

**Under Variable Optical and Electrical Conditions**

This notebook implements a 1D steady-state drift-diffusion / carrier-continuity model of a
monolayer MoS2 photodetector, solved numerically with `scipy.integrate.solve_bvp`. Only
NumPy, SciPy, Pandas and Matplotlib are used (no proprietary TCAD/DFT tools).

**Governing equations** (steady state, `d/dt = 0`, SI units throughout):

- Electrons: `0 = Dn * d2n/dx2 - mu_n * E * dn/dx + G - R`
- Holes:     `0 = Dp * d2p/dx2 + mu_p * E * dp/dx + G - R`
- Einstein relation: `D = mu * kB * T / q`
- Uniform lateral field: `E = V / L`
- Linear excess-carrier recombination: `R = (carrier - equilibrium_carrier) / tau`

Run the cells **top to bottom**. Runtime for the full notebook (including the 2D heatmap and
optimization) is on the order of 1-2 minutes on a standard Colab CPU runtime.

> ⚠️ **Before using results in a paper**: several transport/optical parameters below are
> placeholders flagged `REQUIRES LITERATURE VERIFICATION` (mobility, lifetime, absorbance,
> IQE, equilibrium carrier density, device geometry). Replace them with values you can cite.
> The bandgap (~1.8 eV), excitonic feature (~600 nm) and Mo-S bond length (2.409 Å) are the
> literature-supported values you supplied and are used as given.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from scipy.optimize import minimize
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

## SECTION 1 — Parameters (`initialize_parameters`)

In [ ]:
# ==============================================================================
# SECTION 1: PARAMETERS
# ==============================================================================

def initialize_parameters():
    """
    Central parameter dictionary. SI units are used internally everywhere.

    Two parameter classes are distinguished explicitly:

      (A) LITERATURE-SUPPORTED VALUES -- taken directly from the values you
          supplied (Consensus AI search on monolayer MoS2).

      (B) PARAMETERS REQUIRING LITERATURE VERIFICATION -- no value for these
          was supplied. Representative, order-of-magnitude placeholders
          (commonly seen in TMD photodetector literature) are used so the
          code runs, but YOU MUST replace them with values you can cite
          before using any results in a paper. Each is tagged
          "# REQUIRES LITERATURE VERIFICATION".
    """
    p = {}

    # ---- physical constants (exact, SI) --------------------------------
    p['q']  = 1.602176634e-19      # C
    p['kB'] = 1.380649e-23         # J/K
    p['h']  = 6.62607015e-34       # J s
    p['c']  = 2.99792458e8         # m/s

    # ---- operating condition -------------------------------------------
    p['T'] = 300.0                 # K, room temperature (user-adjustable)

    # ---- (A) literature-supported MoS2 values you supplied --------------
    p['Eg_eV']                    = 1.8      # optical bandgap, eV (~1.7-1.8 eV monolayer; 1.8 eV used)
    p['lambda_exciton_nm']        = 600.0    # excitonic/optical absorption feature, nm
    p['bond_length_MoS_angstrom'] = 2.409    # Mo-S bond length, Angstrom (structural reference; not used
                                              # directly in the transport equations, kept for documentation)

    # ---- (B) PARAMETERS REQUIRING LITERATURE VERIFICATION ---------------
    # Transport
    p['mu_n_cm2Vs'] = 30.0     # electron mobility, cm^2/(V s)   # REQUIRES LITERATURE VERIFICATION
    p['mu_p_cm2Vs'] = 15.0     # hole mobility, cm^2/(V s)       # REQUIRES LITERATURE VERIFICATION
    p['tau_n']      = 1.0e-9   # electron minority/excess lifetime, s   # REQUIRES LITERATURE VERIFICATION
    p['tau_p']      = 1.0e-9   # hole minority/excess lifetime, s       # REQUIRES LITERATURE VERIFICATION

    # Geometry
    p['thickness_nm'] = 0.65   # monolayer thickness, nm                # REQUIRES LITERATURE VERIFICATION
    p['L_channel_um'] = 1.0    # source-drain channel length, um        # REQUIRES LITERATURE VERIFICATION (device-specific)
    p['W_channel_um'] = 5.0    # channel width, um                      # REQUIRES LITERATURE VERIFICATION (device-specific)

    # Optical response
    p['absorbance']         = 0.07   # baseline fraction of incident light absorbed (~5-10% typical
                                      # for monolayer MoS2 is widely reported order-of-magnitude)  # REQUIRES LITERATURE VERIFICATION
    p['exciton_enhancement'] = 1.5   # peak absorbance multiplier at the excitonic feature    # REQUIRES LITERATURE VERIFICATION
    p['exciton_width_nm']    = 25.0  # Gaussian width of the excitonic absorption feature, nm # REQUIRES LITERATURE VERIFICATION
    p['IQE'] = 0.8                    # internal quantum efficiency (fraction of absorbed photons
                                       # producing a collected carrier pair)                   # REQUIRES LITERATURE VERIFICATION

    # Equilibrium (dark) carrier populations -- monolayer MoS2 is typically
    # slightly n-type (sulfur-vacancy-derived donors), so p0 << n0 is assumed.
    p['n_s0_m2'] = 1.0e16   # equilibrium background *sheet* electron density, m^-2   # REQUIRES LITERATURE VERIFICATION
    p['p_s0_m2'] = 1.0e12   # equilibrium background *sheet* hole density, m^-2       # REQUIRES LITERATURE VERIFICATION

    # ---- numerics --------------------------------------------------------
    p['N_grid'] = 100         # initial BVP mesh points along the channel
    p['bvp_tol'] = 1.0e-5     # relative tolerance for solve_bvp (loosened for speed;
                              # tightening below ~1e-6 makes the boundary-layer regions
                              # very expensive to resolve without changing plotted results)

    # ---- derived quantities (SI) ------------------------------------------
    p['L_m']  = p['L_channel_um'] * 1e-6
    p['W_m']  = p['W_channel_um'] * 1e-6
    p['t_m']  = p['thickness_nm'] * 1e-9

    p['mu_n'] = p['mu_n_cm2Vs'] * 1e-4     # m^2/(V s)
    p['mu_p'] = p['mu_p_cm2Vs'] * 1e-4     # m^2/(V s)

    # Einstein relation: D = mu * kB * T / q
    p['D_n'] = p['mu_n'] * p['kB'] * p['T'] / p['q']
    p['D_p'] = p['mu_p'] * p['kB'] * p['T'] / p['q']

    # Convert equilibrium sheet densities to *volumetric* densities so they
    # are compatible with the standard 3D drift-diffusion form of the
    # continuity equation (the monolayer is treated as a thin slab of
    # thickness t_m -- a standard simplification for quasi-2D materials).
    p['n0'] = p['n_s0_m2'] / p['t_m']      # m^-3
    p['p0'] = p['p_s0_m2'] / p['t_m']      # m^-3

    return p

## SECTION 2 — Optics: photon energy, spectral absorbance, generation rate

In [ ]:
# ==============================================================================
# SECTION 2: OPTICS  -  photon energy, spectral absorbance, generation rate
# ==============================================================================

def calculate_photon_energy(wavelength_m, params):
    """Photon energy E = hc/lambda. Returns (E_joules, E_eV)."""
    E_J = params['h'] * params['c'] / wavelength_m
    E_eV = E_J / params['q']
    return E_J, E_eV


def spectral_absorbance(wavelength_nm, params):
    """
    Simple, literature-defensible absorbance model:
      - zero below the optical bandgap (no interband absorption)
      - baseline 'absorbance' above the gap
      - a Gaussian enhancement centered on the reported excitonic feature
    All shape parameters are flagged REQUIRES LITERATURE VERIFICATION.
    """
    wavelength_nm = np.atleast_1d(wavelength_nm).astype(float)
    _, E_eV = calculate_photon_energy(wavelength_nm * 1e-9, params)

    base = params['absorbance']
    bump = (params['exciton_enhancement'] - 1.0) * np.exp(
        -0.5 * ((wavelength_nm - params['lambda_exciton_nm']) / params['exciton_width_nm']) ** 2
    )
    A = base * (1.0 + bump)
    A = np.where(E_eV < params['Eg_eV'], 0.0, A)   # sub-gap photons: negligible absorption
    return A if A.size > 1 else float(A[0])


def calculate_generation_rate(P_opt_W, wavelength_nm, params):
    """
    Spatially-uniform photogeneration rate G0 [m^-3 s^-1].

    Because the monolayer thickness (~0.65 nm) is orders of magnitude
    smaller than any optical absorption length, generation is treated as
    uniform through the thickness and along the channel (full-channel
    illumination) rather than following a Beer-Lambert depth profile.
    """
    wavelength_m = wavelength_nm * 1e-9
    E_photon_J, _ = calculate_photon_energy(wavelength_m, params)

    illum_area = params['L_m'] * params['W_m']      # top-down illuminated area, m^2
    intensity = P_opt_W / illum_area                # W/m^2
    photon_flux = intensity / E_photon_J             # photons / (m^2 s)

    A = spectral_absorbance(wavelength_nm, params)
    absorbed_flux = photon_flux * A                  # photons absorbed / (m^2 s)

    G0 = params['IQE'] * absorbed_flux / params['t_m']   # m^-3 s^-1
    return float(G0)

## SECTION 3 — Transport: steady-state drift-diffusion solve (`solve_bvp`)

In [ ]:
# ==============================================================================
# SECTION 3: TRANSPORT  -  steady-state drift-diffusion solve (solve_bvp)
# ==============================================================================
#
# Steady-state (d/dt = 0) forms of the two governing continuity equations:
#
#   electrons:  0 = Dn n'' - mu_n E n' + G - R   =>  n'' = ( mu_n E n' - G + R) / Dn
#   holes:      0 = Dp p'' + mu_p E p' + G - R   =>  p'' = (-mu_p E p' - G + R) / Dp
#
# with R = (carrier - equilibrium_carrier) / tau  (linear excess-carrier
# recombination) and ohmic-contact (Dirichlet) boundary conditions fixing the
# carrier density at its dark equilibrium value at both contacts.

def solve_transport(V_bias, G0, params, carrier='electron'):
    """
    Solve the steady-state 1D drift-diffusion equation for one carrier type.

    Numerically, the equation is solved for the *excess* carrier density
    (delta_n = n - n_equilibrium) rather than n itself. n_equilibrium is
    typically many orders of magnitude larger than the illumination-induced
    excess, so solving for delta_n directly (instead of the full n, which
    would force solve_bvp's relative tolerance to resolve a tiny perturbation
    riding on a huge baseline) is both faster and numerically well-conditioned.
    The full carrier density n = delta_n + n_equilibrium is reconstructed
    at the end.

    Returns
    -------
    x_fine   : ndarray, position, m
    n_fine   : ndarray, carrier density, m^-3 (clipped >= 0, stability check)
    dndx_fine: ndarray, carrier density gradient, m^-4
    sol      : the raw solve_bvp solution object (for diagnostics)
    """
    L = params['L_m']
    E_field = V_bias / L                      # E = V / L

    if carrier == 'electron':
        mu, D, n_eq, tau, sign = params['mu_n'], params['D_n'], params['n0'], params['tau_n'], +1.0
    elif carrier == 'hole':
        mu, D, n_eq, tau, sign = params['mu_p'], params['D_p'], params['p0'], params['tau_p'], -1.0
    else:
        raise ValueError("carrier must be 'electron' or 'hole'")

    x_mesh = np.linspace(0.0, L, params['N_grid'])

    def odes(x, y):
        # y[0] = delta_n = n - n_eq ; y[1] = d(delta_n)/dx = dn/dx
        delta_n = y[0]
        ddelta_ndx = y[1]
        R = delta_n / tau                      # (n - n_eq)/tau, n_eq cancels
        d2ndx2 = (sign * mu * E_field * ddelta_ndx - G0 + R) / D
        return np.vstack([ddelta_ndx, d2ndx2])

    def bc(ya, yb):
        return np.array([ya[0], yb[0]])        # delta_n = 0 at both ohmic contacts

    y_guess = np.zeros((2, x_mesh.size))
    y_guess[0, :] = G0 * tau                   # rough uniform excess-carrier estimate

    sol = solve_bvp(odes, bc, x_mesh, y_guess, max_nodes=20000, tol=params['bvp_tol'])
    if not sol.success:
        warnings.warn(f"solve_bvp did not fully converge for {carrier} "
                       f"(V={V_bias:.3g} V, G0={G0:.3g}): {sol.message}")

    x_fine = np.linspace(0.0, L, 400)
    y_fine = sol.sol(x_fine)
    n_fine = y_fine[0] + n_eq
    n_fine = np.clip(n_fine, 0.0, None)        # STABILITY CHECK: forbid negative carrier density
    dndx_fine = y_fine[1]
    return x_fine, n_fine, dndx_fine, sol

## SECTION 4 — Current density and terminal current

In [ ]:
# ==============================================================================
# SECTION 4: CURRENT
# ==============================================================================

def calculate_current(x, n, dndx, p, dpdx, V_bias, params):
    """
    Electron and hole current densities (drift + diffusion), and the
    terminal current after converting from current density to current
    using the channel cross-section (width x thickness).

        J_n = q ( mu_n n E + Dn dn/dx )
        J_p = q ( mu_p p E - Dp dp/dx )
    """
    E_field = V_bias / params['L_m']
    q = params['q']

    Jn = q * (params['mu_n'] * n * E_field + params['D_n'] * dndx)
    Jp = q * (params['mu_p'] * p * E_field - params['D_p'] * dpdx)
    Jtotal = Jn + Jp

    # steady-state current is (ideally) position-independent; average over x
    Jn_avg, Jp_avg, Jtot_avg = np.mean(Jn), np.mean(Jp), np.mean(Jtotal)

    cross_section_area = params['W_m'] * params['t_m']
    I_total = Jtot_avg * cross_section_area

    return {'Jn': Jn, 'Jp': Jp, 'Jtotal': Jtotal,
            'Jn_avg': Jn_avg, 'Jp_avg': Jp_avg, 'Jtotal_avg': Jtot_avg,
            'I_total': I_total}

## SECTION 5 — Simulation driver (`run_simulation`, `calculate_responsivity`)

In [ ]:
# ==============================================================================
# SECTION 5: SIMULATION DRIVER
# ==============================================================================

def run_simulation(V_bias, P_opt_W, wavelength_nm, params, compute_dark=True):
    """
    Run a matched pair of dark / illuminated simulations at a given bias,
    optical power and wavelength. Returns a dictionary with carrier
    profiles, currents, photocurrent and responsivity.
    """
    G0_illum = calculate_generation_rate(P_opt_W, wavelength_nm, params)

    # --- illuminated ---
    xe, n_illum, dndx_illum, sol_ne = solve_transport(V_bias, G0_illum, params, 'electron')
    xh, p_illum, dpdx_illum, sol_ph = solve_transport(V_bias, G0_illum, params, 'hole')
    cur_illum = calculate_current(xe, n_illum, dndx_illum, p_illum, dpdx_illum, V_bias, params)

    result = {
        'V_bias': V_bias, 'P_opt': P_opt_W, 'wavelength_nm': wavelength_nm,
        'G0_illum': G0_illum,
        'x': xe,
        'n_illum': n_illum, 'p_illum': p_illum,
        'I_illum': cur_illum['I_total'],
        'Jtotal_illum': cur_illum['Jtotal'],
    }

    if compute_dark:
        xe_d, n_dark, dndx_dark, _ = solve_transport(V_bias, 0.0, params, 'electron')
        xh_d, p_dark, dpdx_dark, _ = solve_transport(V_bias, 0.0, params, 'hole')
        cur_dark = calculate_current(xe_d, n_dark, dndx_dark, p_dark, dpdx_dark, V_bias, params)

        result['n_dark'] = n_dark
        result['p_dark'] = p_dark
        result['I_dark'] = cur_dark['I_total']
        result['Jtotal_dark'] = cur_dark['Jtotal']
        result['I_photo'] = result['I_illum'] - result['I_dark']
        result['responsivity'] = (result['I_photo'] / P_opt_W) if P_opt_W > 0 else np.nan

    return result


def calculate_responsivity(I_photo, P_opt_W):
    """Responsivity R = I_photo / P_opt, in A/W."""
    P_opt_W = np.asarray(P_opt_W, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        R = np.where(P_opt_W > 0, np.asarray(I_photo) / P_opt_W, np.nan)
    return R

## SECTION 6 — Parameter sweeps (bias, power, wavelength, 2D heatmap)

In [ ]:
# ==============================================================================
# SECTION 6: PARAMETER SWEEPS
# ==============================================================================

def sweep_bias(V_array, P_opt_W, wavelength_nm, params):
    I_dark, I_illum, I_photo, Resp = [], [], [], []
    for V in V_array:
        r = run_simulation(V, P_opt_W, wavelength_nm, params, compute_dark=True)
        I_dark.append(r['I_dark']); I_illum.append(r['I_illum'])
        I_photo.append(r['I_photo']); Resp.append(r['responsivity'])
    return pd.DataFrame({'V_bias': V_array, 'I_dark': I_dark, 'I_illum': I_illum,
                          'I_photo': I_photo, 'responsivity': Resp})


def sweep_power(P_array_W, V_bias, wavelength_nm, params):
    dark = run_simulation(V_bias, P_array_W[0], wavelength_nm, params, compute_dark=True)
    I_dark_ref = dark['I_dark']
    I_illum, I_photo, Resp = [], [], []
    for P in P_array_W:
        G0 = calculate_generation_rate(P, wavelength_nm, params)
        xe, n_i, dndx_i, _ = solve_transport(V_bias, G0, params, 'electron')
        xh, p_i, dpdx_i, _ = solve_transport(V_bias, G0, params, 'hole')
        cur = calculate_current(xe, n_i, dndx_i, p_i, dpdx_i, V_bias, params)
        I_illum.append(cur['I_total'])
        I_photo.append(cur['I_total'] - I_dark_ref)
        Resp.append((cur['I_total'] - I_dark_ref) / P if P > 0 else np.nan)
    return pd.DataFrame({'P_opt_W': P_array_W, 'I_dark': I_dark_ref,
                          'I_illum': I_illum, 'I_photo': I_photo, 'responsivity': Resp})


def sweep_wavelength(wavelength_array_nm, V_bias, P_opt_W, params):
    I_dark, I_illum, I_photo, Resp = [], [], [], []
    dark = run_simulation(V_bias, P_opt_W, wavelength_array_nm[0], params, compute_dark=True)
    I_dark_ref = dark['I_dark']
    for wl in wavelength_array_nm:
        G0 = calculate_generation_rate(P_opt_W, wl, params)
        xe, n_i, dndx_i, _ = solve_transport(V_bias, G0, params, 'electron')
        xh, p_i, dpdx_i, _ = solve_transport(V_bias, G0, params, 'hole')
        cur = calculate_current(xe, n_i, dndx_i, p_i, dpdx_i, V_bias, params)
        I_illum.append(cur['I_total'])
        I_photo.append(cur['I_total'] - I_dark_ref)
        Resp.append((cur['I_total'] - I_dark_ref) / P_opt_W if P_opt_W > 0 else np.nan)
    return pd.DataFrame({'wavelength_nm': wavelength_array_nm, 'I_dark': I_dark_ref,
                          'I_illum': I_illum, 'I_photo': I_photo, 'responsivity': Resp})


def responsivity_heatmap(wavelength_array_nm, V_array, P_opt_W, params):
    """2D sweep: responsivity(wavelength, bias). Returns a matrix [len(V), len(wl)]."""
    R_matrix = np.zeros((len(V_array), len(wavelength_array_nm)))
    for i, V in enumerate(V_array):
        dark = run_simulation(V, P_opt_W, wavelength_array_nm[0], params, compute_dark=True)
        I_dark_ref = dark['I_dark']
        for j, wl in enumerate(wavelength_array_nm):
            G0 = calculate_generation_rate(P_opt_W, wl, params)
            xe, n_i, dndx_i, _ = solve_transport(V, G0, params, 'electron')
            xh, p_i, dpdx_i, _ = solve_transport(V, G0, params, 'hole')
            cur = calculate_current(xe, n_i, dndx_i, p_i, dpdx_i, V, params)
            I_photo = cur['I_total'] - I_dark_ref
            R_matrix[i, j] = I_photo / P_opt_W if P_opt_W > 0 else np.nan
    return R_matrix

## SECTION 7 — Optimization of photoresponse

In [ ]:
# ==============================================================================
# SECTION 7: OPTIMIZATION
# ==============================================================================

def optimize_photoresponse(params, P_opt_W,
                            V_bounds=(0.05, 5.0), wl_bounds=(450.0, 700.0)):
    """
    Basic optimization: find (V_bias, wavelength) that maximizes responsivity,
    via bounded local minimization of -R (L-BFGS-B), seeded from a coarse
    grid search to avoid obviously poor local optima.
    """
    # coarse grid seed
    V_seed = np.linspace(*V_bounds, 6)
    wl_seed = np.linspace(*wl_bounds, 6)
    best = (-np.inf, None)
    for V in V_seed:
        for wl in wl_seed:
            G0 = calculate_generation_rate(P_opt_W, wl, params)
            xe, n_i, dndx_i, _ = solve_transport(V, G0, params, 'electron')
            xh, p_i, dpdx_i, _ = solve_transport(V, G0, params, 'hole')
            cur = calculate_current(xe, n_i, dndx_i, p_i, dpdx_i, V, params)
            dark = run_simulation(V, P_opt_W, wl, params, compute_dark=True)
            R = (cur['I_total'] - dark['I_dark']) / P_opt_W
            if R > best[0]:
                best = (R, (V, wl))

    def neg_R(x):
        V, wl = x
        wl = np.clip(wl, *wl_bounds)
        V = np.clip(V, *V_bounds)
        G0 = calculate_generation_rate(P_opt_W, wl, params)
        xe, n_i, dndx_i, _ = solve_transport(V, G0, params, 'electron')
        xh, p_i, dpdx_i, _ = solve_transport(V, G0, params, 'hole')
        cur = calculate_current(xe, n_i, dndx_i, p_i, dpdx_i, V, params)
        dark = run_simulation(V, P_opt_W, wl, params, compute_dark=True)
        R = (cur['I_total'] - dark['I_dark']) / P_opt_W
        return -R

    res = minimize(neg_R, x0=np.array(best[1]), method='L-BFGS-B',
                    bounds=[V_bounds, wl_bounds])

    V_opt, wl_opt = res.x
    R_opt = -res.fun
    return {'V_opt': V_opt, 'wavelength_opt': wl_opt, 'responsivity_opt': R_opt,
            'scipy_result': res, 'grid_seed_best': best}

## SECTION 8 — Plotting functions (publication-quality figures)

In [ ]:
# ==============================================================================
# SECTION 8: PLOTS
# ==============================================================================

def plot_carrier_profiles(result, params, title_suffix=""):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    x_um = result['x'] * 1e6
    ax.plot(x_um, result['n_illum'], label='electrons (illuminated)', color='tab:blue')
    ax.plot(x_um, result['p_illum'], label='holes (illuminated)', color='tab:red')
    if 'n_dark' in result:
        ax.plot(x_um, result['n_dark'], '--', label='electrons (dark)', color='tab:blue', alpha=0.6)
        ax.plot(x_um, result['p_dark'], '--', label='holes (dark)', color='tab:red', alpha=0.6)
    ax.set_xlabel('Position, x (µm)')
    ax.set_ylabel('Carrier density (m$^{-3}$)')
    ax.set_yscale('log')
    ax.set_title(f'Carrier concentration vs. position {title_suffix}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def plot_dark_vs_illum(result, params):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    x_um = result['x'] * 1e6
    axes[0].plot(x_um, result['n_dark'], label='dark', color='k')
    axes[0].plot(x_um, result['n_illum'], label='illuminated', color='tab:orange')
    axes[0].set_title('Electron density: dark vs. illuminated')
    axes[0].set_xlabel('x (µm)'); axes[0].set_ylabel('n (m$^{-3}$)')
    axes[0].set_yscale('log'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(x_um, result['p_dark'], label='dark', color='k')
    axes[1].plot(x_um, result['p_illum'], label='illuminated', color='tab:orange')
    axes[1].set_title('Hole density: dark vs. illuminated')
    axes[1].set_xlabel('x (µm)'); axes[1].set_ylabel('p (m$^{-3}$)')
    axes[1].set_yscale('log'); axes[1].legend(); axes[1].grid(alpha=0.3)
    fig.tight_layout()
    return fig


def plot_current_vs_bias(df_bias):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.plot(df_bias['V_bias'], df_bias['I_dark'] * 1e9, 'o-', label='dark current', color='k')
    ax.plot(df_bias['V_bias'], df_bias['I_illum'] * 1e9, 's-', label='illuminated current', color='tab:orange')
    ax.set_xlabel('Bias voltage, V (V)')
    ax.set_ylabel('Current (nA)')
    ax.set_title('Current density (as terminal current) vs. bias')
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def plot_photocurrent_vs_power(df_power):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.loglog(df_power['P_opt_W'] * 1e6, df_power['I_photo'] * 1e9, 'o-', color='tab:green')
    ax.set_xlabel('Optical power (µW)')
    ax.set_ylabel('Photocurrent (nA)')
    ax.set_title('Photocurrent vs. optical power')
    ax.grid(alpha=0.3, which='both')
    fig.tight_layout()
    return fig


def plot_responsivity_vs_wavelength(df_wl):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.plot(df_wl['wavelength_nm'], df_wl['responsivity'], color='tab:purple')
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Responsivity (A/W)')
    ax.set_title('Responsivity vs. wavelength')
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def plot_responsivity_vs_bias(df_bias):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.plot(df_bias['V_bias'], df_bias['responsivity'], color='tab:brown')
    ax.set_xlabel('Bias voltage, V (V)')
    ax.set_ylabel('Responsivity (A/W)')
    ax.set_title('Responsivity vs. bias')
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def plot_responsivity_heatmap(wavelength_array_nm, V_array, R_matrix, opt_point=None):
    fig, ax = plt.subplots(figsize=(6.5, 5))
    extent = [wavelength_array_nm.min(), wavelength_array_nm.max(), V_array.min(), V_array.max()]
    im = ax.imshow(R_matrix, origin='lower', aspect='auto', extent=extent, cmap='viridis')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Responsivity (A/W)')
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Bias voltage, V (V)')
    ax.set_title('Responsivity heatmap: wavelength vs. bias')
    if opt_point is not None:
        ax.plot(opt_point[1], opt_point[0], marker='*', color='red', markersize=16,
                markeredgecolor='white', label='optimum')
        ax.legend(loc='upper right')
    fig.tight_layout()
    return fig


def plot_optimization_result(opt_result, wavelength_array_nm, V_array, R_matrix):
    fig = plot_responsivity_heatmap(wavelength_array_nm, V_array, R_matrix,
                                     opt_point=(opt_result['V_opt'], opt_result['wavelength_opt']))
    ax = fig.axes[0]
    ax.set_title(f"Optimum: V={opt_result['V_opt']:.2f} V, "
                 f"λ={opt_result['wavelength_opt']:.0f} nm, "
                 f"R={opt_result['responsivity_opt']:.3g} A/W")
    return fig

## Demonstration and Results

The cells below run the model with representative operating conditions and generate every
figure requested for the study. Change `params[...]` values (Section 1) to your
literature-verified numbers and re-run.

In [ ]:
params = initialize_parameters()

print("Derived quantities (SI units):")
for k in ['mu_n', 'mu_p', 'D_n', 'D_p', 'n0', 'p0', 'L_m', 'W_m', 't_m']:
    print(f"  {k:8s} = {params[k]:.6g}")

### 1-2. Carrier concentration vs. position — dark vs. illuminated

In [ ]:
V_demo   = 1.0      # V, applied bias
P_demo   = 1e-6      # W, incident optical power (1 uW)
wl_demo  = 600.0     # nm, wavelength (at the excitonic feature)

result_demo = run_simulation(V_demo, P_demo, wl_demo, params, compute_dark=True)

print(f"Dark current        I_dark  = {result_demo['I_dark']:.4e} A")
print(f"Illuminated current I_illum = {result_demo['I_illum']:.4e} A")
print(f"Photocurrent        I_photo = {result_demo['I_photo']:.4e} A")
print(f"Responsivity        R       = {result_demo['responsivity']:.4e} A/W")

fig1 = plot_carrier_profiles(result_demo, params,
                              title_suffix=f"(V={V_demo} V, P={P_demo*1e6:.1f} µW, λ={wl_demo:.0f} nm)")
plt.show()

fig2 = plot_dark_vs_illum(result_demo, params)
plt.show()

### 3 & 6. Current density vs. bias, and responsivity vs. bias

In [ ]:
V_array = np.linspace(0.05, 3.0, 15)
df_bias = sweep_bias(V_array, P_opt_W=P_demo, wavelength_nm=wl_demo, params=params)
df_bias

In [ ]:
fig3 = plot_current_vs_bias(df_bias)
plt.show()

fig4 = plot_responsivity_vs_bias(df_bias)
plt.show()

### 4. Photocurrent vs. optical power

In [ ]:
P_array = np.linspace(0.1e-6, 10e-6, 15)   # 0.1 - 10 uW
df_power = sweep_power(P_array, V_bias=V_demo, wavelength_nm=wl_demo, params=params)

fig5 = plot_photocurrent_vs_power(df_power)
plt.show()
df_power

### 5. Responsivity vs. wavelength

In [ ]:
wl_array = np.linspace(450, 700, 20)   # nm; model absorbs only above the optical bandgap
df_wl = sweep_wavelength(wl_array, V_bias=V_demo, P_opt_W=P_demo, params=params)

fig6 = plot_responsivity_vs_wavelength(df_wl)
plt.show()
df_wl

### 7. Responsivity heatmap: wavelength vs. bias

In [ ]:
wl_grid = np.linspace(450, 700, 14)
V_grid  = np.linspace(0.05, 3.0, 12)

R_matrix = responsivity_heatmap(wl_grid, V_grid, P_opt_W=P_demo, params=params)

fig7 = plot_responsivity_heatmap(wl_grid, V_grid, R_matrix)
plt.show()

### 8. Basic optimization of the photoresponse

In [ ]:
opt_result = optimize_photoresponse(params, P_opt_W=P_demo,
                                     V_bounds=(0.05, 3.0), wl_bounds=(450.0, 700.0))

print(f"Optimum bias:       V*  = {opt_result['V_opt']:.3f} V")
print(f"Optimum wavelength: λ*  = {opt_result['wavelength_opt']:.1f} nm")
print(f"Maximum responsivity: R* = {opt_result['responsivity_opt']:.4g} A/W")

fig8 = plot_optimization_result(opt_result, wl_grid, V_grid, R_matrix)
plt.show()

## Summary table of key results

In [ ]:
summary = pd.DataFrame({
    "Quantity": ["Dark current (V=1V)", "Photocurrent (V=1V, P=1uW, 600nm)",
                 "Responsivity (V=1V, P=1uW, 600nm)", "Optimum bias", "Optimum wavelength",
                 "Maximum responsivity (grid+L-BFGS-B)"],
    "Value": [f"{result_demo['I_dark']:.3e} A", f"{result_demo['I_photo']:.3e} A",
              f"{result_demo['responsivity']:.3e} A/W", f"{opt_result['V_opt']:.3f} V",
              f"{opt_result['wavelength_opt']:.1f} nm", f"{opt_result['responsivity_opt']:.3e} A/W"]
})
summary

## Notes and limitations

- This is a **decoupled** drift-diffusion model: the electric field is fixed as `E = V/L`
  (no self-consistent Poisson solve, no band bending / Schottky contacts).
- Recombination uses a single linear excess-carrier lifetime (`R = Δn/τ`), not full
  Shockley-Read-Hall statistics with trap energies.
- The monolayer is treated as an optically thin slab (uniform photogeneration through the
  0.65 nm thickness) rather than resolving a Beer-Lambert absorption depth profile — appropriate
  because the absorption length in MoS2 vastly exceeds the monolayer thickness.
- Photoconductive gain mechanisms reported in some experimental monolayer MoS2 devices
  (trap-mediated gain, giving responsivities >> 1 A/W) are **not** included; this model
  captures primary photogeneration + drift-diffusion collection only. If your device shows
  gain, you will need to add a trap/gating model to reproduce it quantitatively.